# 05. Targeted 6-Type Augmentation & Sequential ID Renaming

This notebook provides a complete, isolated workflow for processing new KTP webcam captures added to the training set:
- **`real_ktp`**: 6 new raw images
- **`fake_print`**: 8 new raw images

### Objectives
1. **Rename** raw `WIN_...` files to consecutive numeric IDs continuing from the highest existing ID in each class folder.
2. **Generate exactly 6 augmentations** per original image (1 Brightness, 1 Blur, 1 Rotation, 1 Center Crop, 1 Monochrome, 1 Contrast).
3. **Update Manifests**: Register all new samples into `datasets/split/train/labels.csv` and `datasets/metadata/train_labels.csv`.
4. **Preserve Isolation**: Zero changes to existing training files (`0001` through `0576`) and zero changes to the `test/` partition.

In [ ]:
import os
import re
import json
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Path configuration (relative to /notebooks directory)
TRAIN_DIR = Path("../datasets/split/train").resolve()
METADATA_DIR = Path("../datasets/metadata").resolve()

print(f"📁 Train directory   : {TRAIN_DIR}")
print(f"📋 Metadata directory: {METADATA_DIR}")

## 1. Augmentation Transformations
The 6 targeted transformations matching our project specifications:
1. **Brightness**: Factor 1.25 (scaled RGB intensity)
2. **Gaussian Blur**: 7x7 kernel
3. **Rotation**: +5.0 degrees with BORDER_REFLECT_101 border padding
4. **Center Crop**: Scale 0.85 (15% zoom-in, resized back to original shape)
5. **Monochrome**: 3-channel grayscale / B&W photocopy simulation
6. **Contrast**: Factor 1.30 centered around midpoint 128

In [ ]:
def augment_brightness(img_bgr: np.ndarray, factor: float = 1.25) -> np.ndarray:
    scaled = img_bgr.astype(np.float32) * factor
    return np.clip(scaled, 0, 255).astype(np.uint8)

def augment_blur(img_bgr: np.ndarray, ksize: tuple = (7, 7)) -> np.ndarray:
    return cv2.GaussianBlur(img_bgr, ksize, 0)

def augment_rotation(img_bgr: np.ndarray, angle: float = 5.0) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    center = (w / 2.0, h / 2.0)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img_bgr, matrix, (w, h), borderMode=cv2.BORDER_REFLECT_101)

def augment_center_crop(img_bgr: np.ndarray, scale: float = 0.85) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    crop_w, crop_h = int(w * scale), int(h * scale)
    x1 = (w - crop_w) // 2
    y1 = (h - crop_h) // 2
    cropped = img_bgr[y1 : y1 + crop_h, x1 : x1 + crop_w]
    return cv2.resize(cropped, (w, h), interpolation=cv2.INTER_LINEAR)

def augment_monochrome(img_bgr: np.ndarray) -> np.ndarray:
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def augment_contrast(img_bgr: np.ndarray, factor: float = 1.30) -> np.ndarray:
    adjusted = 128.0 + factor * (img_bgr.astype(np.float32) - 128.0)
    return np.clip(adjusted, 0, 255).astype(np.uint8)

AUGMENTATION_TYPES = [
    ("bright", "Brightness (1.25x)", lambda img: augment_brightness(img, 1.25)),
    ("blur", "Gaussian Blur (7x7)", lambda img: augment_blur(img, (7, 7))),
    ("rot", "Rotation (+5.0 deg)", lambda img: augment_rotation(img, 5.0)),
    ("crop", "Center Crop (85% zoom)", lambda img: augment_center_crop(img, 0.85)),
    ("mono", "Monochrome Grayscale", lambda img: augment_monochrome(img)),
    ("contrast", "Contrast (1.30x)", lambda img: augment_contrast(img, 1.30)),
]
print("Registered 6 augmentation functions!")

## 2. ID Inspection & Augmentation Execution
Finds the maximum existing numeric ID per class folder and sequentially generates the renamed files and their augmentations.

In [ ]:
def get_highest_id(folder: Path) -> int:
    ids = []
    if folder.exists():
        for f in folder.iterdir():
            if f.is_file():
                match = re.match(r"^(\d{4})_", f.name)
                if match:
                    ids.append(int(match.group(1)))
    return max(ids) if ids else 0

targets = {
    "real_ktp": {"dir": TRAIN_DIR / "real_ktp", "label_id": 0, "pattern": "WIN_20261103_08*.jpg"},
    "fake_print": {"dir": TRAIN_DIR / "fake_print", "label_id": 2, "pattern": "WIN_20261103_08*.jpg"},
}

manifest = {}
new_csv_records = []

for class_name, cfg in targets.items():
    folder = cfg["dir"]
    label_id = cfg["label_id"]
    highest_id = get_highest_id(folder)
    start_id = highest_id + 1
    current_id = start_id
    win_files = sorted(list(folder.glob(cfg["pattern"])))
    print(f"\nClass '{class_name}': highest existing ID = {highest_id:04d}, starting next at {start_id:04d}")
    print(f"Found {len(win_files)} new source captures")
    for src_file in win_files:
        orig_filename = src_file.name
        img_bgr = cv2.imread(str(src_file))
        if img_bgr is None:
            continue
        orig_id = current_id
        renamed_name = f"{orig_id:04d}_{class_name}.jpg"
        renamed_path = folder / renamed_name
        src_file.unlink()
        cv2.imwrite(str(renamed_path), img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 95])
        manifest[renamed_name] = {"source_win_filename": orig_filename, "role": "original", "class": class_name}
        new_csv_records.append({"filename": renamed_name, "relative_path": f"train/{class_name}/{renamed_name}", "class_name": class_name, "label_id": label_id})
        current_id += 1
        for aug_key, aug_desc, aug_func in AUGMENTATION_TYPES:
            aug_id = current_id
            aug_name = f"{aug_id:04d}_{class_name}.jpg"
            aug_path = folder / aug_name
            transformed = aug_func(img_bgr)
            cv2.imwrite(str(aug_path), transformed, [cv2.IMWRITE_JPEG_QUALITY, 95])
            manifest[aug_name] = {"source_win_filename": orig_filename, "parent": renamed_name, "role": "augmentation", "type": aug_key}
            new_csv_records.append({"filename": aug_name, "relative_path": f"train/{class_name}/{aug_name}", "class_name": class_name, "label_id": label_id})
            current_id += 1
    print(f"Finished '{class_name}': IDs {start_id:04d} -> {current_id - 1:04d}")

## 3. Save Manifest & Update labels.csv
Appends all new sample records to `labels.csv` and writes the audit JSON manifest.

In [ ]:
manifest_path = METADATA_DIR / "new_captures_augmentation_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(f"Manifest saved: {manifest_path}")

if new_csv_records:
    new_df = pd.DataFrame(new_csv_records)
    for p in [TRAIN_DIR / "labels.csv", METADATA_DIR / "train_labels.csv"]:
        if p.exists():
            existing_df = pd.read_csv(p)
            combined_df = pd.concat([existing_df, new_df], ignore_index=True).drop_duplicates(subset=["filename"])
            combined_df.to_csv(p, index=False)
            print(f"Updated {p.name}: {len(existing_df)} -> {len(combined_df)} rows")

## 4. Visual Verification Grid
Shows a 7-column comparative preview: Original Capture followed by all 6 augmentations.

In [ ]:
for cls_name, prefix_range in [("real_ktp", range(505, 512)), ("fake_print", range(577, 584))]:
    folder = TRAIN_DIR / cls_name
    orig_id = prefix_range.start
    orig_path = folder / f"{orig_id:04d}_{cls_name}.jpg"
    if not orig_path.exists():
        continue
    fig, axes = plt.subplots(1, 7, figsize=(22, 3.5))
    for idx, f_id in enumerate(prefix_range):
        img_path = folder / f"{f_id:04d}_{cls_name}.jpg"
        if img_path.exists():
            img = cv2.imread(str(img_path))
            axes[idx].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
            title = f"ORIGINAL\n{f_id:04d}" if idx == 0 else f"AUG #{idx}\n{f_id:04d}"
            axes[idx].set_title(title, fontsize=8)
        axes[idx].axis("off")
    plt.suptitle(f"{cls_name.upper()} Sample Augmentation Sequence", fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.show()